# EmoFace: train AlexNet on FER-2013 (Google Colab)

This notebook trains the model that the EmoFace website uses. When it finishes you download one file, `alexnet_emotion.pth`, and put it in the `models/` folder of your VS Code project.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Have your Kaggle API token ready: Kaggle > Settings > API > *Create New Token* (downloads `kaggle.json`).
3. Run the cells from top to bottom. Training takes roughly 15 to 40 minutes on a T4.

Set `PRETRAINED = False` in the settings cell and run everything again to train AlexNet from scratch, so your report can compare both approaches.

In [ ]:
import os, time, copy, glob
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageOps, ImageFilter
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# ---- Settings you can change ------------------------------------------------
PRETRAINED = True      # True = ImageNet-pretrained AlexNet (fine-tune). False = train from scratch.
EPOCHS     = 30
BATCH_SIZE = 32
LR         = 1e-4      # learning rate for AlexNet's convolution layers
HEAD_LR    = 5e-4      # larger rate for the new fully connected layers
AUTOCONTRAST = True    # stretch each face's contrast (website does the same: fixes dim or harsh webcam light)
LABEL_SMOOTHING = 0.1  # stops the model becoming over-confident on FER-2013's noisy labels
PATIENCE   = 7         # stop early when validation accuracy has not improved for this many epochs
VAL_SPLIT  = 0.10
SEED       = 42
# -----------------------------------------------------------------------------

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU: switch the runtime to GPU")

## 1. Get FER-2013 from Kaggle

In [ ]:
from google.colab import files

kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(kaggle_json):
    print("Choose the kaggle.json file you downloaded from Kaggle")
    uploaded = files.upload()
    os.makedirs(os.path.dirname(kaggle_json), exist_ok=True)
    with open(kaggle_json, "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod(kaggle_json, 0o600)

!pip -q install kaggle
!kaggle datasets download -d msambare/fer2013 -p /content/data --unzip -q
!ls /content/data

If the download fails, open the dataset page on Kaggle (search "FER-2013" by *msambare*), accept its terms once while signed in, then run the cell again. Any FER-2013 copy works as long as it has `train` and `test` folders that contain one sub-folder per emotion.

In [ ]:
CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

def find_split(root, name):
    """Find the folder called `name` that holds the seven emotion folders."""
    for path in glob.glob(os.path.join(root, "**", name), recursive=True):
        if os.path.isdir(path) and set(CLASSES) <= set(os.listdir(path)):
            return path
    raise FileNotFoundError(f"Could not find a '{name}' folder with the seven emotion folders inside /content/data")

train_dir = find_split("/content/data", "train")
test_dir  = find_split("/content/data", "test")
print("train:", train_dir)
print("test :", test_dir)

for split_dir in (train_dir, test_dir):
    counts = {c: len(os.listdir(os.path.join(split_dir, c))) for c in CLASSES}
    print(os.path.basename(split_dir), counts, "total", sum(counts.values()))

## 2. Preprocessing and data loaders

These are exactly the steps the website applies to a camera photo: grayscale, contrast stretch, resize to 227 x 227, ImageNet normalisation.

Training images also get random flips, shifts, soft focus, lighting changes and noise, so the model learns to cope with **webcam** pictures and not only with the clean FER-2013 photos. (A model trained only on clean FER-2013 often answers "neutral" for every webcam picture.)

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def to_gray(img):
    return img.convert("L")

def stretch_contrast(img):
    return ImageOps.autocontrast(img, cutoff=1) if AUTOCONTRAST else img

def add_noise(t):
    return t + torch.randn_like(t) * 0.15 if torch.rand(1).item() < 0.25 else t

# Training: the intensity changes come BEFORE the contrast stretch, so the model always sees
# faces prepared the same way as at prediction time.
train_tf = transforms.Compose([
    transforms.Lambda(to_gray),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=12, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.2))], p=0.3),   # soft webcam focus
    transforms.ColorJitter(brightness=0.4, contrast=0.4),                              # lighting changes
    transforms.Lambda(stretch_contrast),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((227, 227), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.Lambda(add_noise),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),   # glasses, hands, hair over the face
])
eval_tf = transforms.Compose([
    transforms.Lambda(to_gray),
    transforms.Lambda(stretch_contrast),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((227, 227), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_train = datasets.ImageFolder(train_dir, train_tf)   # augmented view
full_val   = datasets.ImageFolder(train_dir, eval_tf)    # clean view of the same files
test_ds    = datasets.ImageFolder(test_dir, eval_tf)
assert full_train.classes == CLASSES, full_train.classes

order = np.random.permutation(len(full_train))
n_val = int(VAL_SPLIT * len(order))
val_idx, train_idx = order[:n_val], order[n_val:]
train_ds, val_ds = Subset(full_train, train_idx), Subset(full_val, val_idx)

kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=(device.type == "cuda"))
train_loader = DataLoader(train_ds, shuffle=True, **kw)
val_loader   = DataLoader(val_ds, shuffle=False, **kw)
test_loader  = DataLoader(test_ds, shuffle=False, **kw)
print(f"train {len(train_ds)} | validation {len(val_ds)} | test {len(test_ds)}")

# FER-2013 is imbalanced (disgust is rare), so weight the loss by inverse class frequency.
counts = np.bincount(np.array(full_train.targets)[train_idx], minlength=len(CLASSES))
class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32).to(device)
print("class weights:", dict(zip(CLASSES, class_weights.cpu().numpy().round(2))))

## 3. AlexNet with a 7-class output layer

In [ ]:
def build_alexnet(num_classes=7, pretrained=True):
    weights = models.AlexNet_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.alexnet(weights=weights)
    model.classifier[6] = nn.Linear(4096, num_classes)   # 1000 ImageNet classes -> 7 emotions
    return model

model = build_alexnet(len(CLASSES), PRETRAINED).to(device)
print("Pretrained start:", PRETRAINED)
print(model.classifier)

## 4. Train

The loss is cross entropy (softmax is applied inside it, and again at prediction time in the website). The model with the best **validation** accuracy is kept.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
if PRETRAINED:
    # pretrained convolution layers need only gentle adjustment; the new layers learn faster
    optimizer = torch.optim.AdamW([
        {"params": model.features.parameters(), "lr": LR},
        {"params": model.classifier.parameters(), "lr": HEAD_LR},
    ], weight_decay=1e-4)
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                out = model(x)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            seen += y.size(0)
    return total_loss / seen, correct / seen

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_acc, best_state, stale = -1.0, None, 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, True)
    vl, va = run_epoch(val_loader, False)
    scheduler.step()
    for k, v in zip(history, (tl, vl, ta, va)):
        history[k].append(v)
    flag = ""
    if va > best_acc:
        best_acc, best_state, stale, flag = va, copy.deepcopy(model.state_dict()), 0, "  <- best"
    else:
        stale += 1
    print(f"epoch {epoch:02d}/{EPOCHS} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} | {time.time()-t0:.0f}s{flag}")
    if stale >= PATIENCE:
        print(f"Stopping early: no improvement for {PATIENCE} epochs.")
        break

model.load_state_dict(best_state)
print(f"Best validation accuracy: {best_acc:.3f}")

## 5. Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_acc"], label="train"); ax[0].plot(history["val_acc"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(history["train_loss"], label="train"); ax[1].plot(history["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

## 6. Evaluate, and remove any bias towards one class

Predictions use the face **and its mirror image** averaged, exactly like the website.

FER-2013 has many "neutral" and "happy" faces, so a model can lean towards them. The cell below measures how often each class is predicted and finds a small correction per class on the **validation** images (never the test images). The correction is saved in the model file and the website applies it.

In [ ]:
@torch.no_grad()
def collect_logits(loader):
    model.eval()
    out, ys = [], []
    for x, y in loader:
        x = x.to(device)
        out.append(((model(x) + model(torch.flip(x, dims=[3]))) / 2).float().cpu())
        ys.append(y)
    return torch.cat(out).numpy(), torch.cat(ys).numpy()

val_logits, val_y = collect_logits(val_loader)
test_logits, test_y = collect_logits(test_loader)

def macro_f1(logits, y, bias):
    return f1_score(y, (logits + bias).argmax(1), average="macro")

def accuracy(logits, y, bias):
    return float(((logits + bias).argmax(1) == y).mean())

bias = np.zeros(len(CLASSES), dtype=np.float32)
best = macro_f1(val_logits, val_y, bias)
for _ in range(3):                         # a few passes of coordinate search
    for c in range(len(CLASSES)):
        for g in np.linspace(-1.5, 1.5, 31):
            trial = bias.copy(); trial[c] = g
            f = macro_f1(val_logits, val_y, trial)
            if f > best + 1e-6:
                best, bias = f, trial

# keep the correction only if it does not cost noticeable overall accuracy
if accuracy(val_logits, val_y, bias) < accuracy(val_logits, val_y, 0) - 0.02:
    bias = np.zeros_like(bias)
bias = bias.astype(np.float32)
print("Class correction (logit bias):", dict(zip(CLASSES, bias.round(2))))
print()
print(f"Validation accuracy  without correction {accuracy(val_logits, val_y, 0):.4f} | with {accuracy(val_logits, val_y, bias):.4f}")
print(f"Test accuracy        without correction {accuracy(test_logits, test_y, 0):.4f} | with {accuracy(test_logits, test_y, bias):.4f}   <- report this one")
print()

y_true = test_y
y_pred = (test_logits + bias).argmax(1)
y_plain = test_logits.argmax(1)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0))

share = lambda p: np.bincount(p, minlength=len(CLASSES)) / len(p)
print("How often each class is predicted, compared with how often it really occurs (test set):")
print(f"{'class':10s} {'really':>8s} {'predicted, no correction':>26s} {'predicted, corrected':>22s}")
for c, name in enumerate(CLASSES):
    print(f"{name:10s} {share(y_true)[c]:8.1%} {share(y_plain)[c]:26.1%} {share(y_pred)[c]:22.1%}")

cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(7), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(7), CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (row-normalised)")
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="white" if cm_norm[i, j] > .5 else "black", fontsize=8)
fig.colorbar(im); plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

## 6b. Webcam check

The website takes pictures with a webcam, which are softer, flatter and noisier than FER-2013. This cell makes the test images look like that (blur, low contrast, lighting shift, noise) and measures the model again. Accuracy should drop only a little, and the share of "neutral" answers should stay close to the real share. If almost everything becomes neutral, increase `EPOCHS` or leave `AUTOCONTRAST = True` and train again.

In [ ]:
def degrade(img):
    img = img.filter(ImageFilter.GaussianBlur(radius=np.random.uniform(0.6, 1.4)))
    a = np.asarray(img).astype(np.float32)
    a = (a - 128) * np.random.uniform(0.45, 0.8) + 128 + np.random.uniform(-40, 40)
    a = a + np.random.normal(0, 7, a.shape)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

class WebcamLikeFolder(datasets.ImageFolder):
    def __getitem__(self, i):
        path, target = self.samples[i]
        return self.transform(degrade(Image.open(path).convert("L"))), target

webcam_ds = WebcamLikeFolder(test_dir, eval_tf)
webcam_loader = DataLoader(webcam_ds, shuffle=False, **kw)
wc_logits, wc_y = collect_logits(webcam_loader)
wc_pred = (wc_logits + bias).argmax(1)
ni = CLASSES.index("neutral")
print(f"Accuracy on webcam-like test images : {accuracy(wc_logits, wc_y, bias):.4f}   (clean test images: {accuracy(test_logits, test_y, bias):.4f})")
print(f"Predicted as neutral                : {share(wc_pred)[ni]:.1%}   (really neutral: {share(wc_y)[ni]:.1%}; clean images predicted neutral: {share(y_pred)[ni]:.1%})")

## 7. Save the model and download it

In [ ]:
checkpoint = {
    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
    "classes": CLASSES,
    "input_size": 227,
    "autocontrast": AUTOCONTRAST,         # the website reads these three settings
    "logit_bias": [float(b) for b in bias],
    "pretrained_start": PRETRAINED,
    "val_acc": best_acc,
    "test_acc": float(accuracy(test_logits, test_y, bias)),
}
torch.save(checkpoint, "alexnet_emotion.pth")

# Prove the file loads the same way the website loads it.
check = torch.load("alexnet_emotion.pth", map_location="cpu", weights_only=False)
reload = build_alexnet(len(check["classes"]), pretrained=False)
reload.load_state_dict(check["state_dict"])
print("Saved and reloaded OK. Classes:", check["classes"])

files.download("alexnet_emotion.pth")
files.download("training_curves.png")
files.download("confusion_matrix.png")

**Next:** put `alexnet_emotion.pth` into the `models/` folder of the EmoFace project in VS Code and restart the server (`python app.py`).